In [47]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/European_Bank.csv")

In [48]:
print("Total Customers:", len(df))
print("Churned Customers:", df["Exited"].sum())
print("Retained Customers:", (df["Exited"] == 0).sum())

churn_rate = df["Exited"].mean() * 100

print(f"Overall Churn Rate: {churn_rate:.2f}%")

Total Customers: 10000
Churned Customers: 2037
Retained Customers: 7963
Overall Churn Rate: 20.37%


In [49]:
df["Exited"].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [50]:
df["Exited"].value_counts(normalize=True) * 100

Exited
0    79.63
1    20.37
Name: proportion, dtype: float64

In [51]:
geo_churn = pd.crosstab(
    df["Geography"],
    df["Exited"],
    margins=True
)

geo_churn

Exited,0,1,All
Geography,,,
France,4204,810,5014
Germany,1695,814,2509
Spain,2064,413,2477
All,7963,2037,10000


In [52]:
geo_churn_rate = (
    df.groupby("Geography")["Exited"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

geo_churn_rate

Geography
Germany    32.443204
Spain      16.673395
France     16.154767
Name: Exited, dtype: float64

In [53]:
geo_customer_count = df["Geography"].value_counts()

geo_customer_count

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [54]:
geo_age_churn = (
    df.groupby(["Geography", "Age"])["Exited"]
      .mean()
      .mul(100)
      .reset_index()
)

geo_age_churn.head(20)

,Geography,Age,Exited
0,France,18,9.090909
1,France,19,0.000000
2,France,20,0.000000
3,France,21,3.225806
4,France,22,13.333333
5,France,23,4.444444
6,France,24,5.479452
7,France,25,2.898551
8,France,26,6.194690
9,France,27,5.172414


In [55]:
def age_group(age):
    if age < 30:
        return "<30"
    elif age <= 45:
        return "30-45"
    elif age <= 60:
        return "46-60"
    else:
        return "60+"

df["AgeGroup"] = df["Age"].apply(age_group)

df["AgeGroup"].value_counts()

AgeGroup
30-45    6248
46-60    1647
<30      1641
60+       464
Name: count, dtype: int64

In [56]:
age_churn = (
    df.groupby("AgeGroup")["Exited"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

age_churn

AgeGroup
46-60    51.123254
60+      24.784483
30-45    15.300896
<30       7.556368
Name: Exited, dtype: float64

In [57]:
geo_age_churn = (
    df.groupby(["Geography", "AgeGroup"])["Exited"]
      .mean()
      .mul(100)
      .reset_index()
)

geo_age_churn


,Geography,AgeGroup,Exited
0,France,30-45,11.868687
1,France,46-60,45.794393
2,France,60+,19.913420
3,France,<30,5.196305
4,Germany,30-45,25.295664
5,Germany,46-60,67.330677
6,Germany,60+,38.938053
7,Germany,<30,12.634409
8,Spain,30-45,12.516046
9,Spain,46-60,40.656566


In [58]:
geo_age_churn = (
    df.groupby(["Geography", "AgeGroup"])["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

geo_age_churn["ChurnRate"] = geo_age_churn["mean"] * 100

geo_age_churn = geo_age_churn.drop(columns="mean")

geo_age_churn

,Geography,AgeGroup,count,sum,ChurnRate
0,France,30-45,3168,376,11.868687
1,France,46-60,749,343,45.794393
2,France,60+,231,46,19.913420
3,France,<30,866,45,5.196305
4,Germany,30-45,1522,385,25.295664
5,Germany,46-60,502,338,67.330677
6,Germany,60+,113,44,38.938053
7,Germany,<30,372,47,12.634409
8,Spain,30-45,1558,195,12.516046
9,Spain,46-60,396,161,40.656566


In [59]:
geo_age_churn.sort_values(
    "ChurnRate",
    ascending=False
)

,Geography,AgeGroup,count,sum,ChurnRate
5,Germany,46-60,502,338,67.330677
1,France,46-60,749,343,45.794393
9,Spain,46-60,396,161,40.656566
6,Germany,60+,113,44,38.938053
4,Germany,30-45,1522,385,25.295664
10,Spain,60+,120,25,20.833333
2,France,60+,231,46,19.913420
7,Germany,<30,372,47,12.634409
8,Spain,30-45,1558,195,12.516046
0,France,30-45,3168,376,11.868687


In [60]:
geo_age_matrix = pd.pivot_table(
    df,
    values="Exited",
    index="AgeGroup",
    columns="Geography",
    aggfunc="mean"
) * 100

geo_age_matrix

Geography,France,Germany,Spain
AgeGroup,,,
30-45,11.868687,25.295664,12.516046
46-60,45.794393,67.330677,40.656566
60+,19.913420,38.938053,20.833333
<30,5.196305,12.634409,7.940447


In [61]:
df["CreditScore"].describe()

count    10000.000000
mean       650.528800
std         96.653299
min        350.000000
25%        584.000000
50%        652.000000
75%        718.000000
max        850.000000
Name: CreditScore, dtype: float64

In [62]:
df["CreditScore"].min(), df["CreditScore"].max()

(np.int64(350), np.int64(850))

In [63]:
df["CreditScoreBand"] = pd.cut(
    df["CreditScore"],
    bins=[-np.inf, 584, 718, np.inf],
    labels=["Low", "Medium", "High"]
)
df["CreditScoreBand"].value_counts().sort_index()

CreditScoreBand
Low       2534
Medium    5003
High      2463
Name: count, dtype: int64

In [64]:
credit_churn = (
    df.groupby("CreditScoreBand", observed=True)["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

credit_churn["ChurnRate"] = credit_churn["mean"] * 100

credit_churn = credit_churn.drop(columns="mean")

credit_churn

,CreditScoreBand,count,sum,ChurnRate
0,Low,2534,556,21.941594
1,Medium,5003,981,19.608235
2,High,2463,500,20.300447


In [65]:
df["Balance"].describe()

count     10000.000000
mean      76485.889288
std       62397.405202
min           0.000000
25%           0.000000
50%       97198.540000
75%      127644.240000
max      250898.090000
Name: Balance, dtype: float64

In [66]:
df["Balance"].quantile([0.25, 0.50, 0.75])

0.25         0.00
0.50     97198.54
0.75    127644.24
Name: Balance, dtype: float64

In [67]:
print("Zero-balance customers:", (df["Balance"] == 0).sum())
print("Non-zero balance customers:", (df["Balance"] > 0).sum())

Zero-balance customers: 3617
Non-zero balance customers: 6383


In [68]:
def balance_segment(balance):
    if balance == 0:
        return "Zero-balance"
    elif balance <= 97198.54:
        return "Low-balance"
    else:
        return "High-balance"

df["BalanceSegment"] = df["Balance"].apply(balance_segment)
df["BalanceSegment"].value_counts()

BalanceSegment
High-balance    5000
Zero-balance    3617
Low-balance     1383
Name: count, dtype: int64

In [69]:
balance_churn = (
    df.groupby("BalanceSegment")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

balance_churn["ChurnRate"] = balance_churn["mean"] * 100

balance_churn = balance_churn.drop(columns="mean")

balance_churn

,BalanceSegment,count,sum,ChurnRate
0,High-balance,5000,1249,24.980000
1,Low-balance,1383,288,20.824295
2,Zero-balance,3617,500,13.823611


In [70]:
balance_churn.sort_values(
    "ChurnRate",
    ascending=False
)

,BalanceSegment,count,sum,ChurnRate
0,High-balance,5000,1249,24.980000
1,Low-balance,1383,288,20.824295
2,Zero-balance,3617,500,13.823611


In [71]:
churned_customers = df[df["Exited"] == 1]

print("Total churned customers:", len(churned_customers))

print(
    "Total balance held by churned customers:",
    churned_customers["Balance"].sum()
)

print(
    "Average balance of churned customers:",
    churned_customers["Balance"].mean()
)

print(
    "Median balance of churned customers:",
    churned_customers["Balance"].median()
)

Total churned customers: 2037
Total balance held by churned customers: 185588094.63
Average balance of churned customers: 91108.53933726068
Median balance of churned customers: 109349.29


In [72]:
balance_profile = (
    df.groupby("Exited")["Balance"]
      .agg(["count", "mean", "median", "sum"])
)

balance_profile

,count,mean,median,sum
Exited,,,,
0,7963,72745.296779,92072.68,5.792708e+08
1,2037,91108.539337,109349.29,1.855881e+08


In [73]:
high_value = df[df["BalanceSegment"] == "High-balance"]

high_value_churn_ratio = high_value["Exited"].mean() * 100

print("High-value customers:", len(high_value))
print("High-value churners:", high_value["Exited"].sum())
print(f"High-Value Churn Ratio: {high_value_churn_ratio:.2f}%")

High-value customers: 5000
High-value churners: 1249
High-Value Churn Ratio: 24.98%


In [74]:
high_value_churners = high_value[high_value["Exited"] == 1]

print(
    "High-value churner balance:",
    high_value_churners["Balance"].sum()
)

print(
    "Average balance of high-value churners:",
    high_value_churners["Balance"].mean()
)

High-value churner balance: 163242872.31
Average balance of high-value churners: 130698.85693354683


In [75]:
high_value_churn_contribution = (
    len(high_value_churners) / df["Exited"].sum()
) * 100

print(
    f"High-value churn contribution: "
    f"{high_value_churn_contribution:.2f}%"
)

High-value churn contribution: 61.32%


In [76]:
df["EstimatedSalary"].describe()

count     10000.000000
mean     100090.239881
std       57510.492818
min          11.580000
25%       51002.110000
50%      100193.915000
75%      149388.247500
max      199992.480000
Name: EstimatedSalary, dtype: float64

In [77]:
df["EstimatedSalary"].quantile([0.25, 0.50, 0.75])

0.25     51002.1100
0.50    100193.9150
0.75    149388.2475
Name: EstimatedSalary, dtype: float64

In [78]:
salary_profile = (
    df.groupby("Exited")["EstimatedSalary"]
      .agg(["count", "mean", "median", "sum"])
)

salary_profile

,count,mean,median,sum
Exited,,,,
0,7963,99738.391772,99645.04,7.942168e+08
1,2037,101465.677531,102460.84,2.066856e+08


In [79]:
churned_profile = (
    df[df["Exited"] == 1][
        ["Balance", "EstimatedSalary"]
    ].describe()
)

churned_profile

,Balance,EstimatedSalary
count,2037.000000,2037.000000
mean,91108.539337,101465.677531
std,58360.794816,57912.418071
min,0.000000,11.580000
25%,38340.020000,51907.720000
50%,109349.290000,102460.840000
75%,131433.330000,152422.910000
max,250898.090000,199808.100000


In [80]:
high_value_salary = (
    high_value_churners["EstimatedSalary"]
    .agg(["count", "mean", "median", "sum"])
)

high_value_salary

count     1.249000e+03
mean      1.002037e+05
median    1.012401e+05
sum       1.251544e+08
Name: EstimatedSalary, dtype: float64

In [81]:
df[["Balance", "EstimatedSalary", "Exited"]].corr()

,Balance,EstimatedSalary,Exited
Balance,1.000000,0.012797,0.118533
EstimatedSalary,0.012797,1.000000,0.012097
Exited,0.118533,0.012097,1.000000


In [82]:
financial_profile = (
    df.groupby("Exited")[["Balance", "EstimatedSalary"]]
      .agg(["mean", "median"])
)

financial_profile

Balance            EstimatedSalary           
                mean     median            mean     median
Exited                                                    
0       72745.296779   92072.68    99738.391772   99645.04
1       91108.539337  109349.29   101465.677531  102460.84

In [83]:
financial_segment_profile = (
    df.groupby("BalanceSegment")[
        ["Balance", "EstimatedSalary", "Exited"]
    ].agg({
        "Balance": ["mean", "median"],
        "EstimatedSalary": ["mean", "median"],
        "Exited": ["mean", "sum"]
    })
)

financial_segment_profile

Balance            EstimatedSalary              \
                         mean     median            mean      median   
BalanceSegment                                                         
High-balance    131033.174840  127646.04   100612.032478  100825.255   
Low-balance      79315.270195   82729.49   101098.121135  102606.920   
Zero-balance         0.000000       0.00    98983.559549   98618.790   

                  Exited        
                    mean   sum  
BalanceSegment                  
High-balance    0.249800  1249  
Low-balance     0.208243   288  
Zero-balance    0.138236   500

In [84]:
df["Tenure"].describe()

count    10000.000000
mean         5.012800
std          2.892174
min          0.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         10.000000
Name: Tenure, dtype: float64

In [85]:
df["Tenure"].value_counts().sort_index()

Tenure
0      413
1     1035
2     1048
3     1009
4      989
5     1012
6      967
7     1028
8     1025
9      984
10     490
Name: count, dtype: int64

In [86]:
def tenure_group(tenure):
    if tenure <= 3:
        return "New"
    elif tenure <= 7:
        return "Mid-term"
    else:
        return "Long-term"

df["TenureGroup"] = df["Tenure"].apply(tenure_group)
df["TenureGroup"].value_counts()

TenureGroup
Mid-term     3996
New          3505
Long-term    2499
Name: count, dtype: int64

In [87]:
tenure_churn = (
    df.groupby("TenureGroup")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

tenure_churn["ChurnRate"] = tenure_churn["mean"] * 100

tenure_churn = tenure_churn.drop(columns="mean")

tenure_churn

,TenureGroup,count,sum,ChurnRate
0,Long-term,2499,511,20.448179
1,Mid-term,3996,785,19.644645
2,New,3505,741,21.141227


In [88]:
tenure_churn.sort_values("ChurnRate", ascending=False)

,TenureGroup,count,sum,ChurnRate
2,New,3505,741,21.141227
0,Long-term,2499,511,20.448179
1,Mid-term,3996,785,19.644645


In [90]:
df["IsActiveMember"].value_counts()

IsActiveMember
1    5151
0    4849
Name: count, dtype: int64

In [91]:
activity_churn = (
    df.groupby("IsActiveMember")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

activity_churn["ChurnRate"] = activity_churn["mean"] * 100

activity_churn = activity_churn.drop(columns="mean")

activity_churn

,IsActiveMember,count,sum,ChurnRate
0,0,4849,1302,26.850897
1,1,5151,735,14.269074


In [92]:
df["HasCrCard"].value_counts()

HasCrCard
1    7055
0    2945
Name: count, dtype: int64

In [93]:
card_churn = (
    df.groupby("HasCrCard")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

card_churn["ChurnRate"] = card_churn["mean"] * 100
card_churn = card_churn.drop(columns="mean")

card_churn

,HasCrCard,count,sum,ChurnRate
0,0,2945,613,20.814941
1,1,7055,1424,20.184266


In [94]:
df["NumOfProducts"].value_counts().sort_index()

NumOfProducts
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

In [95]:
product_churn = (
    df.groupby("NumOfProducts")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

product_churn["ChurnRate"] = product_churn["mean"] * 100
product_churn = product_churn.drop(columns="mean")

product_churn

,NumOfProducts,count,sum,ChurnRate
0,1,5084,1409,27.714398
1,2,4590,348,7.581699
2,3,266,220,82.706767
3,4,60,60,100.000000


In [96]:
gender_churn = (
    df.groupby("Gender")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

gender_churn["ChurnRate"] = gender_churn["mean"] * 100
gender_churn = gender_churn.drop(columns="mean")

gender_churn

,Gender,count,sum,ChurnRate
0,Female,4543,1139,25.071539
1,Male,5457,898,16.455928


In [97]:
risk_profile = (
    df.groupby(
        ["Geography", "AgeGroup", "IsActiveMember", "BalanceSegment"]
    )["Exited"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

risk_profile["ChurnRate"] = risk_profile["mean"] * 100

risk_profile = risk_profile.drop(columns="mean")

risk_profile = risk_profile.sort_values(
    "ChurnRate",
    ascending=False
)

risk_profile.head(15)

,Geography,AgeGroup,IsActiveMember,BalanceSegment,count,sum,ChurnRate
12,France,60+,0,High-balance,15,15,100.000000
53,Spain,60+,0,Low-balance,4,4,100.000000
13,France,60+,0,Low-balance,8,8,100.000000
28,Germany,46-60,0,High-balance,237,195,82.278481
32,Germany,60+,0,High-balance,26,21,80.769231
33,Germany,60+,0,Low-balance,5,4,80.000000
54,Spain,60+,0,Zero-balance,9,7,77.777778
47,Spain,46-60,0,Low-balance,26,20,76.923077
29,Germany,46-60,0,Low-balance,39,28,71.794872
52,Spain,60+,0,High-balance,7,5,71.428571


In [98]:
reliable_risk_profiles = (
    risk_profile[risk_profile["count"] >= 50]
    .sort_values("ChurnRate", ascending=False)
)

reliable_risk_profiles.head(15)

,Geography,AgeGroup,IsActiveMember,BalanceSegment,count,sum,ChurnRate
28,Germany,46-60,0,High-balance,237,195,82.278481
46,Spain,46-60,0,High-balance,64,40,62.500000
6,France,46-60,0,High-balance,144,87,60.416667
8,France,46-60,0,Zero-balance,154,92,59.740260
48,Spain,46-60,0,Zero-balance,76,45,59.210526
30,Germany,46-60,1,High-balance,195,105,53.846154
9,France,46-60,1,High-balance,144,51,35.416667
24,Germany,30-45,0,High-balance,616,212,34.415584
10,France,46-60,1,Low-balance,63,21,33.333333
41,Spain,30-45,0,Low-balance,93,30,32.258065


In [99]:
reliable_risk_profiles["ChurnContribution"] = (
    reliable_risk_profiles["sum"] / df["Exited"].sum() * 100
)

reliable_risk_profiles.head(15)

,Geography,AgeGroup,IsActiveMember,BalanceSegment,count,sum,ChurnRate,ChurnContribution
28,Germany,46-60,0,High-balance,237,195,82.278481,9.572901
46,Spain,46-60,0,High-balance,64,40,62.500000,1.963672
6,France,46-60,0,High-balance,144,87,60.416667,4.270987
8,France,46-60,0,Zero-balance,154,92,59.740260,4.516446
48,Spain,46-60,0,Zero-balance,76,45,59.210526,2.209131
30,Germany,46-60,1,High-balance,195,105,53.846154,5.154639
9,France,46-60,1,High-balance,144,51,35.416667,2.503682
24,Germany,30-45,0,High-balance,616,212,34.415584,10.407462
10,France,46-60,1,Low-balance,63,21,33.333333,1.030928
41,Spain,30-45,0,Low-balance,93,30,32.258065,1.472754
